In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler
from collections import Counter

BATCH_SIZE = 64
IMAGE_SIZE = 224
EPOCHS = 30
LR = 0.001

train_dir = r'C:/Emotion_Detection/data/train'
test_dir = r'C:/Emotion_Detection/data/test'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device in use:", device)


Device in use: cuda


TRANSFORM & AUGMENTATION

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [ ]:
train_dataset = ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = ImageFolder(root=test_dir, transform=test_transforms)

emotion_classes = train_dataset.classes
print("Emotion Classes:", emotion_classes)

train_counts = Counter([label for _, label in train_dataset])
test_counts = Counter([label for _, label in test_dataset])
print("Train Distribution:", train_counts)
print("Test Distribution:", test_counts)


Emotion Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train Distribution: Counter({3: 7215, 4: 4965, 5: 4830, 2: 4097, 0: 3995, 6: 3171, 1: 436})
Test Distribution: Counter({3: 1774, 5: 1247, 4: 1233, 2: 1024, 0: 958, 6: 831, 1: 111})


BALANCING

In [ ]:
class_counts = np.array([train_counts[i] for i in range(len(emotion_classes))])
class_weights = 1. / class_counts
sample_weights = [class_weights[label] for _, label in train_dataset]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))


Train batches: 449
Test batches: 113


EFFICIENTNET ARCHITECTURE

In [ ]:
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(emotion_classes)) 
model = model.to(device)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Gopinath/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:13<00:00, 1.57MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)


TRAINING LOOP

In [ ]:
def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, num_epochs, device):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        scheduler.step()
        train_acc = 100 * correct / total
        train_loss /= total

        val_loss, correct, total = 0.0, 0, 0
        model.eval()
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        val_loss /= total

        print(f"Epoch [{epoch}/{num_epochs}] → Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

    return history


In [ ]:

start_time = time.time()
history = train_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=EPOCHS,
    device=device
)
end_time = time.time()
print(f" Training complete in {(end_time - start_time)/60:.2f} minutes.")


Epoch [1/30] → Train Loss: 1.3580, Acc: 55.12% | Val Loss: 1.2688, Acc: 60.77%
Epoch [2/30] → Train Loss: 1.1639, Acc: 65.55% | Val Loss: 1.2221, Acc: 63.26%
Epoch [3/30] → Train Loss: 1.0930, Acc: 68.78% | Val Loss: 1.1823, Acc: 65.51%
Epoch [4/30] → Train Loss: 1.0547, Acc: 70.80% | Val Loss: 1.2019, Acc: 64.63%
Epoch [5/30] → Train Loss: 1.0243, Acc: 72.20% | Val Loss: 1.1680, Acc: 66.48%
Epoch [6/30] → Train Loss: 0.9925, Acc: 73.83% | Val Loss: 1.1394, Acc: 67.75%
Epoch [7/30] → Train Loss: 0.9632, Acc: 75.54% | Val Loss: 1.1917, Acc: 66.01%
Epoch [8/30] → Train Loss: 0.9324, Acc: 77.03% | Val Loss: 1.1830, Acc: 66.63%
Epoch [9/30] → Train Loss: 0.9072, Acc: 78.37% | Val Loss: 1.1699, Acc: 67.72%
Epoch [10/30] → Train Loss: 0.8795, Acc: 80.03% | Val Loss: 1.1598, Acc: 68.03%
Epoch [11/30] → Train Loss: 0.8059, Acc: 83.63% | Val Loss: 1.1605, Acc: 69.82%
Epoch [12/30] → Train Loss: 0.7606, Acc: 85.76% | Val Loss: 1.1661, Acc: 69.78%
Epoch [13/30] → Train Loss: 0.7333, Acc: 87.18% |

KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), 'emotion_model_best.pth')
print("✅ Model saved successfully!")


✅ Model saved successfully!


TESTING WITH UNSEEN DATA

In [ ]:
model.load_state_dict(torch.load('emotion_model_best.pth'))
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
print(f"✅ Test Accuracy: {test_accuracy:.2f}%")


C:\Users\Gopinath\AppData\Local\Temp\ipykernel_22000\3260123544.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('emotion_model_best.pth'

✅ Test Accuracy: 70.55%
